# OceanBench 1/12 degree grid alignment

This notebook opens one forecast start date lazily, compares challenger latitude/longitude coordinates against GLORYS and GLO12 references, and reports how many horizontal cells xarray keeps with its default `inner` arithmetic alignment.

In [ ]:
DATE = "20240103"
TOLERANCE = 1e-4
INCLUDE_GLONET = False
CHECK_GLO12_COMPONENTS = True

In [ ]:
import hashlib
from dataclasses import dataclass
from datetime import datetime
from typing import Iterable

import numpy as np
import pandas as pd
import xarray as xr

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

In [ ]:
@dataclass(frozen=True)
class ZarrDatasetSpec:
    name: str
    url_template: str


@dataclass(frozen=True)
class CopernicusDatasetSpec:
    name: str
    dataset_id: str
    variables: tuple[str, ...]


CHALLENGERS = (
    ZarrDatasetSpec(
        "GLO12_challenger",
        "https://minio.dive.edito.eu/project-oceanbench/public/GLO12/{date}.zarr",
    ),
    ZarrDatasetSpec(
        "XIHE_challenger",
        "https://minio.dive.edito.eu/project-oceanbench/public/XIHE/{date}.zarr",
    ),
    ZarrDatasetSpec(
        "WENHAI_challenger",
        "https://minio.dive.edito.eu/project-oceanbench/public/WENHAI/{date}.zarr",
    ),
    ZarrDatasetSpec(
        "GLONET_challenger",
        "https://minio.dive.edito.eu/project-oceanbench/public/glonet_full_2024/{date}.zarr",
    ),
)

REFERENCES = (
    CopernicusDatasetSpec(
        "GLORYS_ref",
        "cmems_mod_glo_phy_my_0.083deg_P1D-m",
        ("thetao",),
    ),
    CopernicusDatasetSpec(
        "GLO12_ref",
        "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m",
        ("thetao",),
    ),
)

GLO12_REFERENCE_COMPONENTS = (
    CopernicusDatasetSpec(
        "GLO12_ref_thetao",
        "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m",
        ("thetao",),
    ),
    CopernicusDatasetSpec(
        "GLO12_ref_so",
        "cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m",
        ("so",),
    ),
    CopernicusDatasetSpec(
        "GLO12_ref_cur",
        "cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
        ("uo", "vo"),
    ),
    CopernicusDatasetSpec(
        "GLO12_ref_zos",
        "cmems_mod_glo_phy_anfc_0.083deg_P1D-m",
        ("zos",),
    ),
)

LATITUDE_CANDIDATES = ("latitude", "lat")
LONGITUDE_CANDIDATES = ("longitude", "lon")

In [ ]:
def parse_date(date: str) -> pd.Timestamp:
    if len(date) == 8 and date.isdigit():
        return pd.Timestamp(datetime.strptime(date, "%Y%m%d"))
    return pd.Timestamp(date)


def date_key(date: str) -> str:
    return parse_date(date).strftime("%Y%m%d")


def start_end_datetimes(date: str) -> tuple[str, str]:
    start = parse_date(date)
    end = start + pd.Timedelta(days=9)
    return start.strftime("%Y-%m-%dT00:00:00"), end.strftime("%Y-%m-%dT00:00:00")


def zarr_source(spec: ZarrDatasetSpec, date: str) -> str:
    return spec.url_template.format(date=date_key(date))


def open_zarr_dataset(spec: ZarrDatasetSpec, date: str) -> xr.Dataset:
    return xr.open_dataset(zarr_source(spec, date), engine="zarr", chunks={})


def copernicus_source(spec: CopernicusDatasetSpec) -> str:
    return f"{spec.dataset_id} variables={','.join(spec.variables)}"


def open_copernicus_dataset(spec: CopernicusDatasetSpec, date: str) -> xr.Dataset:
    import copernicusmarine

    start_datetime, end_datetime = start_end_datetimes(date)
    return copernicusmarine.open_dataset(
        dataset_id=spec.dataset_id,
        variables=list(spec.variables),
        start_datetime=start_datetime,
        end_datetime=end_datetime,
    )

In [ ]:
def find_coordinate_name(dataset: xr.Dataset, candidates: Iterable[str], standard_name: str) -> str:
    for candidate in candidates:
        if candidate in dataset.coords or candidate in dataset.variables:
            return candidate
    for name in dataset.variables:
        if getattr(dataset[name], "standard_name", None) == standard_name:
            return name
    raise KeyError(f"Could not find {standard_name!r} coordinate in dataset variables: {list(dataset.variables)}")


def coordinate_values(dataset: xr.Dataset, standard_name: str) -> tuple[str, np.ndarray]:
    if standard_name == "latitude":
        coordinate_name = find_coordinate_name(dataset, LATITUDE_CANDIDATES, standard_name)
    elif standard_name == "longitude":
        coordinate_name = find_coordinate_name(dataset, LONGITUDE_CANDIDATES, standard_name)
    else:
        raise ValueError(f"Unsupported coordinate {standard_name!r}")
    return coordinate_name, dataset[coordinate_name].values


def spacing(values: np.ndarray) -> float:
    if len(values) < 2:
        return float("nan")
    return abs(float(values[1] - values[0]))


def coordinate_hash(values: np.ndarray) -> str:
    return hashlib.sha256(values.tobytes()).hexdigest()[:16]


def dataset_grid_summary(name: str, dataset: xr.Dataset, source: str) -> dict[str, object]:
    latitude_name, latitude_values = coordinate_values(dataset, "latitude")
    longitude_name, longitude_values = coordinate_values(dataset, "longitude")
    return {
        "dataset": name,
        "source": source,
        "dims": dict(dataset.sizes),
        "latitude_name": latitude_name,
        "latitude_size": len(latitude_values),
        "latitude_dtype": str(latitude_values.dtype),
        "latitude_first": float(latitude_values[0]),
        "latitude_last": float(latitude_values[-1]),
        "latitude_spacing": spacing(latitude_values),
        "latitude_sha256": coordinate_hash(latitude_values),
        "longitude_name": longitude_name,
        "longitude_size": len(longitude_values),
        "longitude_dtype": str(longitude_values.dtype),
        "longitude_first": float(longitude_values[0]),
        "longitude_last": float(longitude_values[-1]),
        "longitude_spacing": spacing(longitude_values),
        "longitude_sha256": coordinate_hash(longitude_values),
    }

In [ ]:
def coordinate_alignment(
    challenger_dataset: xr.Dataset,
    reference_dataset: xr.Dataset,
    coordinate: str,
    tolerance: float,
) -> dict[str, object]:
    challenger_coordinate_name, challenger_values = coordinate_values(challenger_dataset, coordinate)
    reference_coordinate_name, reference_values = coordinate_values(reference_dataset, coordinate)

    exact_values = np.intersect1d(challenger_values, reference_values)
    common_size = min(len(challenger_values), len(reference_values))
    same_size = len(challenger_values) == len(reference_values)

    positional_difference = challenger_values[:common_size] - reference_values[:common_size]
    positional_mismatch_indexes = np.where(positional_difference != 0)[0]

    first_mismatch = {}
    last_mismatch = {}
    if len(positional_mismatch_indexes):
        i = int(positional_mismatch_indexes[0])
        j = int(positional_mismatch_indexes[-1])
        first_mismatch = {
            "first_mismatch_index": i,
            "first_challenger_value": float(challenger_values[i]),
            "first_reference_value": float(reference_values[i]),
            "first_difference": float(positional_difference[i]),
        }
        last_mismatch = {
            "last_mismatch_index": j,
            "last_challenger_value": float(challenger_values[j]),
            "last_reference_value": float(reference_values[j]),
            "last_difference": float(positional_difference[j]),
        }

    return {
        "coordinate": coordinate,
        "challenger_coordinate_name": challenger_coordinate_name,
        "reference_coordinate_name": reference_coordinate_name,
        "challenger_size": len(challenger_values),
        "reference_size": len(reference_values),
        "exact_common": len(exact_values),
        "ignored_by_coordinate": len(challenger_values) - len(exact_values),
        "exact_ratio": len(exact_values) / len(challenger_values) if len(challenger_values) else float("nan"),
        "same_size": same_size,
        "array_equal": same_size and np.array_equal(challenger_values, reference_values),
        "allclose_tolerance": same_size and np.allclose(challenger_values, reference_values, rtol=0.0, atol=tolerance),
        "max_abs_positional_difference": float(np.max(np.abs(positional_difference))) if common_size else float("nan"),
        "positional_mismatch_count": int(len(positional_mismatch_indexes)),
        **first_mismatch,
        **last_mismatch,
    }


def pair_alignment_summary(
    challenger_name: str,
    challenger_dataset: xr.Dataset,
    reference_name: str,
    reference_dataset: xr.Dataset,
    tolerance: float,
) -> tuple[dict[str, object], list[dict[str, object]]]:
    latitude = coordinate_alignment(challenger_dataset, reference_dataset, "latitude", tolerance)
    longitude = coordinate_alignment(challenger_dataset, reference_dataset, "longitude", tolerance)

    challenger_cells = latitude["challenger_size"] * longitude["challenger_size"]
    exact_common_cells = latitude["exact_common"] * longitude["exact_common"]
    ignored_cells = challenger_cells - exact_common_cells

    summary = {
        "challenger": challenger_name,
        "reference": reference_name,
        "latitude_exact": f"{latitude['exact_common']}/{latitude['challenger_size']}",
        "longitude_exact": f"{longitude['exact_common']}/{longitude['challenger_size']}",
        "cells_used_by_xarray_inner_join": exact_common_cells,
        "challenger_cells": challenger_cells,
        "cells_ignored_by_xarray_inner_join": ignored_cells,
        "cells_used_ratio": exact_common_cells / challenger_cells if challenger_cells else float("nan"),
        "cells_ignored_ratio": ignored_cells / challenger_cells if challenger_cells else float("nan"),
        "array_equal_latitude": latitude["array_equal"],
        "array_equal_longitude": longitude["array_equal"],
        "allclose_latitude": latitude["allclose_tolerance"],
        "allclose_longitude": longitude["allclose_tolerance"],
        "max_abs_latitude_difference": latitude["max_abs_positional_difference"],
        "max_abs_longitude_difference": longitude["max_abs_positional_difference"],
    }
    details = [
        {"challenger": challenger_name, "reference": reference_name, **latitude},
        {"challenger": challenger_name, "reference": reference_name, **longitude},
    ]
    return summary, details

In [ ]:
challengers = {}
sources = {}

for spec in CHALLENGERS:
    if spec.name == "GLONET_challenger" and not INCLUDE_GLONET:
        continue
    challengers[spec.name] = open_zarr_dataset(spec, DATE)
    sources[spec.name] = zarr_source(spec, DATE)

references = {}
for spec in REFERENCES:
    references[spec.name] = open_copernicus_dataset(spec, DATE)
    sources[spec.name] = copernicus_source(spec)

print(f"Date: {date_key(DATE)}")
print(f"xarray arithmetic_join: {xr.get_options()['arithmetic_join']}")
print(f"allclose tolerance: {TOLERANCE:g}")

In [ ]:
grid_summary_df = pd.DataFrame(
    [
        dataset_grid_summary(name, dataset, sources[name])
        for name, dataset in {**challengers, **references}.items()
    ]
)

display(grid_summary_df)

In [ ]:
summary_rows = []
detail_rows = []

for challenger_name, challenger_dataset in challengers.items():
    for reference_name, reference_dataset in references.items():
        summary, details = pair_alignment_summary(
            challenger_name,
            challenger_dataset,
            reference_name,
            reference_dataset,
            TOLERANCE,
        )
        summary_rows.append(summary)
        detail_rows.extend(details)

summary_df = pd.DataFrame(summary_rows)
coordinate_differences_df = pd.DataFrame(detail_rows)

display(summary_df)

In [ ]:
detail_columns = [
    "challenger",
    "reference",
    "coordinate",
    "challenger_coordinate_name",
    "reference_coordinate_name",
    "challenger_size",
    "reference_size",
    "exact_common",
    "ignored_by_coordinate",
    "exact_ratio",
    "array_equal",
    "allclose_tolerance",
    "max_abs_positional_difference",
    "positional_mismatch_count",
    "first_mismatch_index",
    "first_challenger_value",
    "first_reference_value",
    "first_difference",
    "last_mismatch_index",
    "last_challenger_value",
    "last_reference_value",
    "last_difference",
]

display(coordinate_differences_df.reindex(columns=detail_columns))

In [ ]:
if CHECK_GLO12_COMPONENTS:
    component_datasets = {
        spec.name: open_copernicus_dataset(spec, DATE)
        for spec in GLO12_REFERENCE_COMPONENTS
    }
    base_name = "GLO12_ref_thetao"
    base_dataset = component_datasets[base_name]
    component_rows = []
    for component_name, component_dataset in component_datasets.items():
        for coordinate in ("latitude", "longitude"):
            alignment = coordinate_alignment(base_dataset, component_dataset, coordinate, TOLERANCE)
            component_rows.append(
                {
                    "base_component": base_name,
                    "component": component_name,
                    "coordinate": coordinate,
                    "exact_common": alignment["exact_common"],
                    "base_size": alignment["challenger_size"],
                    "array_equal": alignment["array_equal"],
                    "allclose_tolerance": alignment["allclose_tolerance"],
                    "max_abs_positional_difference": alignment["max_abs_positional_difference"],
                }
            )
    component_summary_df = pd.DataFrame(component_rows)
else:
    component_summary_df = pd.DataFrame()

display(component_summary_df)

Interpretation:

- `cells_used_by_xarray_inner_join` is the number of horizontal cells retained before the RMSD spatial mean.
- `cells_ignored_by_xarray_inner_join` is the number of challenger horizontal cells dropped by exact label alignment.
- `array_equal == False` and `allclose_tolerance == True` means the grids are nominally the same, but their float coordinate labels differ enough for xarray to drop cells.